In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="lu0pKfANlkQBo92nc5gR")
project = rf.workspace("mobkr").project("sign-1lqkx-w23ro")
version = project.version(1)
dataset = version.download("folder")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 17.5 MB/s eta 0:00:0000:0100:01m
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-big


Extracting Dataset Version Zip to Sign-1 in folder:: 100%|██████████| 5089/5089 [00:00<00:00, 6460.13it/s]


In [3]:
import os

train_path = "/kaggle/working/Sign-1/train"
val_path   = "/kaggle/working/Sign-1/valid"
test_path  = "/kaggle/working/Sign-1/test"

print("Train classes:", os.listdir(train_path))
print("Damaged count:", len(os.listdir(train_path + "/Damaged")))
print("Healthy count:", len(os.listdir(train_path + "/Healthy")))

Train classes: ['Healthy', 'Damaged']
Damaged count: 2574
Healthy count: 2367


In [4]:
import tensorflow as tf

train_path = "/kaggle/working/Sign-1/train"
val_path   = "/kaggle/working/Sign-1/valid"
test_path  = "/kaggle/working/Sign-1/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

2026-04-29 21:44:49.126715: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777499089.465477      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777499089.568753      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777499090.417290      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777499090.417330      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777499090.417333      57 computation_placer.cc:177] computation placer alr

Found 4941 files belonging to 2 classes.


I0000 00:00:1777499124.865262      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777499124.871507      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 116 files belonging to 2 classes.
Found 30 files belonging to 2 classes.
Classes: ['Damaged', 'Healthy']


In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

In [6]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
])

In [7]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  
inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(2, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop]
)

Epoch 1/50


I0000 00:00:1777499279.169288     145 cuda_dnn.cc:529] Loaded cuDNN version 91002


155/155 ━━━━━━━━━━━━━━━━━━━━ 26s 65ms/step - accuracy: 0.5481 - loss: 0.9406 - val_accuracy: 0.7069 - val_loss: 0.5634
Epoch 2/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 51ms/step - accuracy: 0.6313 - loss: 0.7326 - val_accuracy: 0.7155 - val_loss: 0.5366
Epoch 3/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - accuracy: 0.6717 - loss: 0.6349 - val_accuracy: 0.6983 - val_loss: 0.5288
Epoch 4/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - accuracy: 0.6862 - loss: 0.6120 - val_accuracy: 0.7414 - val_loss: 0.5230
Epoch 5/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.7074 - loss: 0.5692 - val_accuracy: 0.7069 - val_loss: 0.5074
Epoch 6/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.7069 - loss: 0.5678 - val_accuracy: 0.7241 - val_loss: 0.5098
Epoch 7/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.7218 - loss: 0.5378 - val_accuracy: 0.7672 - val_loss: 0.4890
Epoch 8/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.7332 - loss: 0.5545 - val_accuracy: 0.77

In [15]:
base_model.trainable = True

for layer in base_model.layers[:-10]:
    layer.trainable = False

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

In [18]:
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[lr_schedule]
)

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 18s 71ms/step - accuracy: 0.6901 - loss: 0.6570 - val_accuracy: 0.7500 - val_loss: 0.4607 - learning_rate: 1.0000e-05
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.7044 - loss: 0.5945 - val_accuracy: 0.7845 - val_loss: 0.4632 - learning_rate: 1.0000e-05
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.7301 - loss: 0.5662 - val_accuracy: 0.7759 - val_loss: 0.4842 - learning_rate: 1.0000e-05
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.7540 - loss: 0.5344 - val_accuracy: 0.7759 - val_loss: 0.4891 - learning_rate: 3.0000e-06
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.7406 - loss: 0.5308 - val_accuracy: 0.7672 - val_loss: 0.4984 - learning_rate: 3.0000e-06
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.7539 - loss: 0.5220 - val_accuracy: 0.7672 - val_loss: 0.5031 - learning_rate: 9.0000e-07
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/ste

In [19]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.argmax(axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
              precision    recall  f1-score   support

     Damaged       0.68      0.76      0.72        17
     Healthy       0.64      0.54      0.58        13

    accuracy                           0.67        30
   macro avg       0.66      0.65      0.65        30
weighted avg       0.66      0.67      0.66        30



In [20]:
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [21]:
from tensorflow.keras.applications.efficientnet import preprocess_input

inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = preprocess_input(x)

In [24]:
x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(2, activation='softmax')(x)

model_eff = tf.keras.Model(inputs, outputs)

In [25]:
model_eff.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [26]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

In [27]:
history_eff = model_eff.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop, lr_schedule]
)

Epoch 1/50


E0000 00:00:1777499866.443321      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_2_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


155/155 ━━━━━━━━━━━━━━━━━━━━ 25s 98ms/step - accuracy: 0.5675 - loss: 0.7109 - val_accuracy: 0.7845 - val_loss: 0.4411 - learning_rate: 3.0000e-04
Epoch 2/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.7873 - loss: 0.4562 - val_accuracy: 0.8793 - val_loss: 0.3263 - learning_rate: 3.0000e-04
Epoch 3/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.8334 - loss: 0.3801 - val_accuracy: 0.8966 - val_loss: 0.2777 - learning_rate: 3.0000e-04
Epoch 4/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.8643 - loss: 0.3310 - val_accuracy: 0.8966 - val_loss: 0.2510 - learning_rate: 3.0000e-04
Epoch 5/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.8553 - loss: 0.3256 - val_accuracy: 0.8966 - val_loss: 0.2310 - learning_rate: 3.0000e-04
Epoch 6/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.8724 - loss: 0.3047 - val_accuracy: 0.9397 - val_loss: 0.2305 - learning_rate: 3.0000e-04
Epoch 7/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accurac

In [28]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

In [29]:
model_eff.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [31]:
history_eff_ft = model_eff.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10


E0000 00:00:1777500408.870628      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_2_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


155/155 ━━━━━━━━━━━━━━━━━━━━ 29s 103ms/step - accuracy: 0.8436 - loss: 0.3625 - val_accuracy: 0.9310 - val_loss: 0.1926
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 85ms/step - accuracy: 0.8751 - loss: 0.2958 - val_accuracy: 0.9224 - val_loss: 0.1946
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - accuracy: 0.8880 - loss: 0.2618 - val_accuracy: 0.9138 - val_loss: 0.1862
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - accuracy: 0.8942 - loss: 0.2509 - val_accuracy: 0.9310 - val_loss: 0.1754
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - accuracy: 0.9174 - loss: 0.2217 - val_accuracy: 0.9310 - val_loss: 0.1687
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - accuracy: 0.9090 - loss: 0.2243 - val_accuracy: 0.9397 - val_loss: 0.1593
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - accuracy: 0.9172 - loss: 0.2064 - val_accuracy: 0.9397 - val_loss: 0.1523
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 84ms/step - accuracy: 0.9201 - loss: 0.1968 - val_accura

In [32]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model_eff.predict(images)  
    y_true.extend(labels.numpy())
    y_pred.extend(preds.argmax(axis=1))

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
              precision    recall  f1-score   support

     Damaged       0.88      0.88      0.88        17
     Healthy       0.85      0.85      0.85        13

    accuracy                           0.87        30
   macro avg       0.86      0.86      0.86        30
weighted avg       0.87      0.87      0.87        30



In [34]:
model_eff.save("traffic_sign_classifier_effnet.keras")